In [ ]:
import importlib
import sys
import numpy as np
import re
import math

import pandas as pd
import os
import plotly.express as px
import itertools


if os.getcwd().endswith("notebooks"):
    lib_path = "../lib"
else:
    lib_path = "lib"

modules = [("lib", "__init__.py"), ("lib.analysis_utils", "analysis_utils/__init__.py")]

for module_name, module_path in modules:
    spec = importlib.util.spec_from_file_location(module_name, os.path.join(lib_path, module_path))
    module_obj = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module_obj
    spec.loader.exec_module(module_obj)



from lib import PipelineConfig
from lib import analysis_utils

In [ ]:
if "snakemake" in locals():
    feeder_outputs = snakemake.params.feeder_outputs
    no_feeder_outputs = snakemake.params.no_feeder_outputs
    input_paths = snakemake.input
    pipeline_config = PipelineConfig(snakemake.params.config, **snakemake.params.pipeline_kwargs)
    scenario_name = snakemake.wildcards.scenario
    scenario_config = pipeline_config.scenarios[scenario_name]
else:
    feeder_outputs = ["../outputs/routing/outputs/simulated_feeder_transitWithAbstractAccess_200_300_0",
                      "../outputs/routing/outputs/simulated_feeder_transitWithAbstractAccess_400_300_0"]
    no_feeder_outputs = ["../outputs/routing/outputs/simulated_nofeeder_pt"]
    input_paths = feeder_outputs + no_feeder_outputs
    import yaml
    with open("../configs/test_config.yaml") as f:
        pipeline_config = PipelineConfig(yaml.safe_load(f), basedir="../", max_cores=4)
    scenario_name = "feeder"
    scenario_config = pipeline_config.scenarios[scenario_name]
    # raise Exception("This notebook is only snakemake-compatible for now")

In [ ]:
print("Values considered for the radius")
scenario_config.feeders.radiis

In [ ]:
print("Values considered for the frequency")
scenario_config.feeders.frequencies

In [ ]:
print("Values considered for the speeed")
scenario_config.feeders.speeds

In [ ]:
print("List of passed input paths")
[no_feeder_outputs] + feeder_outputs

Each Path is of the form /path/to/something/simulated_{scenario}_transitWithAbstractAccess_{radius}_{frequency}_{speed_index}
- radius: the value of the radius parameter
- frequency: value of the frequency
- speed_index: index of the speed value in the speeds list displayed above (I didn't want to risk passing floats around)

Next we display the path to the outputs of the simulation corresponding to the scenario without feeder service

# What to do next ?

In general we want to measure the relationship between the level of introduction of feeder services and how much better could the trips get relatively to existing public transport.

Some sanity checks, verify that all the scenarios have the same set of trips (person_id, person_trip_id, departure_time, origin, destination). If not, keep only the same ones while we investigate the source of the problem.

Parse the simulation outputs corresponding to each feeder settings and do some analysis.
Maybe the interesting KPI to consider here is the route cost of each trip. It is the one optimized directly by the routing algorithm so I think it makes sense. Here are the parameters:
- `rail_u_h`, `subway_u_h`, `bus_u_h`, `tram_u_h`, `other_u_h`: the marginal utility of time (in hours) spent in respective PT modes. The default value of these parameters is -7.
- `wait_u_h`: the marginal utility of time (in hours) spent waiting for public transport, no matter its mode.  The default value of this parameter is -6.
- `walk_u_h`: the markinal utility of time (in hours) spent walking to/from/between public transport legs.  Its default value is -7.
- `transfer_u`: the marginal utility of a transfer. Its default value is -1.

We can consider the whole chain including feeders (so also considering those transfers) and weighing time spent in a feeder service similarly to the bus. Instead of just keeping the total cost for the trip, we can also have the different components separated.

Then a sanity check we can have is to make sure that no trip sees its routing cost get worse before/after feeder.

Then we can have various graphs with this
- One line plot, feeder radius on the x-axis, total cost on the y axis. If multiple frequencies are considered, we put them in differently coloured lines. If multiple speeds are considered, we can used dashed lines.
- Same plot but instead of total routing cost, we can have the number of trips where the routing cost improves.
- Stacked bar plot, x axis for the feeder radius, y axis for the total cost, colors for the cost components, facet_col for frequency and facet_row for speed.
Then we can check the impact on the usage of existing PT systems (even though we should already kind of have an idea with the cost components)
- A plot showing the number of pt legs per pt mode (rail, subway, tram, feeder) in each setting.
- ...Other analyses

In [ ]:
## Extract feeder parameters from file paths
def extract_parameters(path):
    """Extraire les paramètres des chemins de fichiers de scénarios feeder
    
    Pattern: simulated_{scenario}_transitWithAbstractAccess_{radius}_{frequency}_{speed_index}
    
    Args:
        path (str): Chemin vers le fichier de sortie du scénario
        
    Returns:
        tuple: (radius, frequency, speed, speed_index)
    """
    pattern = r'simulated_.*_transitWithAbstractAccess_(\d+)_(\d+)_(\d+)'
    match = re.search(pattern, path)
    
    if match:
        radius = int(match.group(1))
        frequency = int(match.group(2))
        speed_index = int(match.group(3))
        speed = scenario_config.feeders.speeds[speed_index]
        return radius, frequency, speed, speed_index
    else:
        raise Exception(f"Bad folder name format: {path}")

In [ ]:
# Load data

## No feeder
no_feeder_outputs = no_feeder_outputs[0]

scenario_description_columns = ["scenario_name", "radius", "frequency", "speed"]

params_dict = dict(scenario_name="Baseline", radius=np.nan, frequency=np.nan, speed=np.nan)
nofeeder_legs = analysis_utils.read_simulation_csv(no_feeder_outputs, "eqasim_legs.csv", **params_dict)
nofeeder_pt = analysis_utils.read_simulation_csv(no_feeder_outputs, "eqasim_pt.csv", **params_dict)
nofeeder_trips = analysis_utils.read_simulation_csv(no_feeder_outputs, "eqasim_trips.csv", **params_dict)
nofeeder_routing_costs = analysis_utils.read_simulation_csv(no_feeder_outputs, "pt_routing_costs.csv", **params_dict)

## Feeders
feeders_legs = [pd.read_csv(os.path.join(feeder_output, "eqasim_legs.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_pt = [pd.read_csv(os.path.join(feeder_output, "eqasim_pt.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_trips = [pd.read_csv(os.path.join(feeder_output, "eqasim_trips.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_abstract_access = [pd.read_csv(os.path.join(feeder_output, "eqasim_abstract_access_legs.csv"), sep=";") for feeder_output in feeder_outputs]
feeders_routing_costs = [pd.read_csv(os.path.join(feeder_output, "pt_routing_costs.csv"), sep=";") for feeder_output in feeder_outputs]


feeders_legs = []
feeders_pt = []
feeders_trips = []
feeders_abstract_access = []
feeders_routing_costs = []

feeders_legs_by_scenario = dict()
feeders_pt_by_scenario = dict()
feeders_trips_by_scenario = dict()
feeders_abstract_access_by_scenario = dict()
feeders_routing_costs_by_scenario = dict()

for output_path in feeder_outputs:
    radius, frequency, speed, _ = extract_parameters(output_path)
    scenario_name = f"Feeder R{radius} F{frequency} S{speed:.1f}"
    params_dict = dict(scenario_name=scenario_name, radius=radius, frequency=frequency, speed=speed)

    df_legs = analysis_utils.read_simulation_csv(output_path, "eqasim_legs.csv", **params_dict)
    df_pt_legs = analysis_utils.read_simulation_csv(output_path, "eqasim_pt.csv", **params_dict)
    df_trips = analysis_utils.read_simulation_csv(output_path, "eqasim_trips.csv", **params_dict)
    df_abstract_access = analysis_utils.read_simulation_csv(output_path, "eqasim_abstract_access_legs.csv", **params_dict)
    df_routing_costs = analysis_utils.read_simulation_csv(output_path, "pt_routing_costs.csv", **params_dict)

    for df, l, d in zip([df_legs, df_pt_legs, df_trips, df_abstract_access, df_routing_costs],
                     [feeders_legs, feeders_pt, feeders_trips, feeders_abstract_access, feeders_routing_costs],
                     [feeders_legs_by_scenario, feeders_pt_by_scenario, feeders_trips_by_scenario, feeders_abstract_access_by_scenario, feeders_routing_costs_by_scenario]):
        l.append(df)
        if scenario_name in d:
            raise Exception("Entry for %s already exists" % scenario_name)
        d[scenario_name] = df

feeders_legs = pd.concat(feeders_legs)
feeders_pt = pd.concat(feeders_pt)
feeders_trips = pd.concat(feeders_trips)
feeders_abstract_access = pd.concat(feeders_abstract_access)
feeders_routing_costs = pd.concat(feeders_routing_costs)


all_legs = pd.concat([nofeeder_legs, feeders_legs])
all_pt = pd.concat([nofeeder_pt, feeders_pt])
all_trips = pd.concat([nofeeder_trips, feeders_trips])
all_routing_costs = pd.concat([nofeeder_routing_costs, feeders_routing_costs])

In [ ]:
# Sanity Check
# We mainly compare that the same pairs person_id, trip_id are the same in the trips of all scenarios
# And with the same pairs, we compare some attributes that should also be the same
# (origin and destination coordinates and departure time)

key_columns = ["person_id", "person_trip_id"]
comparison_columns = ["origin_x", "origin_y", "destination_x", "destination_y", "departure_time"]
analysis_columns = key_columns + comparison_columns

summary = []

for scenario_name, df in feeders_trips.groupby("scenario_name"):
    merged = nofeeder_trips[analysis_columns].merge(
        df[analysis_columns],
        on=key_columns,
        suffixes=("_nofeeder", "_feeder"),
        how="outer",
        indicator=True
    )
    only_nofeeder = (merged['_merge'] == 'left_only').sum()
    only_feeder = (merged['_merge'] == 'right_only').sum()
    both = (merged['_merge'] == 'both').sum()
    pct_common = both / len(merged) * 100 if len(merged) > 0 else 0

    common = merged[merged['_merge'] == 'both']
    diffs = {}
    for col in comparison_columns:
        col_nofeeder = f"{col}_nofeeder"
        col_feeder = f"{col}_feeder"
        assert col_nofeeder in common and col_feeder in common
        if col_nofeeder in common and col_feeder in common:
            mask = (~common[col_nofeeder].isna()) & (~common[col_feeder].isna())
            if col in ['origin_x', 'origin_y', 'destination_x', 'destination_y']:
                diff = (abs(common.loc[mask, col_nofeeder] - common.loc[mask, col_feeder]) > 1e-6).sum()
            else:
                diff = (common.loc[mask, col_nofeeder] != common.loc[mask, col_feeder]).sum()
            diffs[col] = diff

    summary.append({
        "scenario": scenario_name,
        "frequency": df["frequency"].unique()[0],
        "speed": df["speed"].unique()[0],
        "radius": df["radius"].unique()[0],
        "total": len(merged),
        "common": both,
        "only_nofeeder": only_nofeeder,
        "only_feeder": only_feeder,
        "pct_common": pct_common,
        "diffs": diffs,
        "nb_diffs": sum(diffs.values()),
    })

df_summary = pd.DataFrame.from_records(summary)

df_summary["radius"] = df_summary["radius"].astype("str")
fig = px.bar(df_summary, x="scenario", y="pct_common", color="radius", facet_col="speed", facet_row="frequency",
             labels=dict(pct_common="% of trips in common with the reference"),
             title="Comparing trips between reference and feeder scenarios according to (person_id, person_trip_id)")
fig.show()


fig = px.bar(df_summary, x="scenario", y="nb_diffs", color="radius", facet_col="speed", facet_row="frequency",
             labels=dict(nb_diffs="Number of differences between reference and feeder scenarios"),
             title="Number of difference in relevant attributes of common trips between reference and feeder scenarios according to (person_id, person_trip_id)")
fig.show()


assert df_summary["pct_common"].min() == 100.0
assert df_summary["nb_diffs"].max() == 0

In [ ]:
# Sanity checks on routing costs
def cost_stats(df, cost_column):
    cost_data = df[cost_column].dropna()
    if len(cost_data) == 0:
        return None
    Q1 = cost_data.quantile(0.25)
    Q3 = cost_data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = cost_data[(cost_data < lower_bound) | (cost_data > upper_bound)]
    return {
        'count': len(cost_data),
        'mean': cost_data.mean(),
        'median': cost_data.median(),
        'std': cost_data.std(),
        'min': cost_data.min(),
        'max': cost_data.max(),
        'outliers(%)': len(outliers) / len(cost_data) * 100,
        'negatives': (cost_data < 0).sum(),
        'zeros': (cost_data == 0).sum()
    }

df = all_routing_costs.groupby("scenario_name").apply(lambda df: pd.Series(cost_stats(df, "routingCost")), include_groups=False)
display(df)
assert df["negatives"].sum() == 0

## Impact of feeders on routing costs
### On the overall population

In [ ]:
# Preparing some data structures this will be used for analysis
df_routing_costs_comparison = []
for scenario_name, df_routing_costs in feeders_routing_costs.groupby("scenario_name"):
    df = nofeeder_routing_costs[["person_id", "trip_id", "routingCost"]].merge(
         df_routing_costs[[c for c in df_routing_costs.columns if c != "mode"]],
         on=["person_id","trip_id"],
         suffixes=("_nofeeder", "_feeder")
    )
    df["delta_abs"] = df["routingCost_feeder"] - df["routingCost_nofeeder"]
    df["delta_pct"] = df["routingCost_feeder"] / df["routingCost_nofeeder"] - 1
    df_routing_costs_comparison.append(df)

df_routing_costs_comparison = pd.concat(df_routing_costs_comparison)

df_routing_costs_comparison["comparison"] = "same"
df_routing_costs_comparison.loc[df_routing_costs_comparison["delta_abs"] < 0, "comparison"] = "better"
df_routing_costs_comparison.loc[df_routing_costs_comparison["delta_abs"] > 0, "comparison"] = "worse"

assert "worse" not in df_routing_costs_comparison["comparison"].unique()

In [ ]:
df_routing_costs_comparison.groupby("scenario_name")["comparison"].value_counts()

In [ ]:
all_routing_costs.groupby("scenario_name").agg(dict(routingCost="mean",
                                                    **{c: "first" for c in scenario_description_columns if c != "scenario_name"})).reset_index()

### On improved trips only

In [ ]:
df_improved = df_routing_costs_comparison[df_routing_costs_comparison["delta_abs"] <0]
df = df_improved.groupby("scenario_name").agg(dict(routingCost_feeder="mean",
                                                   routingCost_nofeeder="mean",
                                                   delta_abs="mean",
                                                   delta_pct="mean",
                                                   **{c: "first" for c in scenario_description_columns}))
df.head()

In [ ]:
def impact_on_improved_routing_line_plot(x, color, facet_col, y, title=None):

    df_plot = df_routing_costs_comparison[df_routing_costs_comparison["delta_abs"] <0].groupby(["scenario_name", "frequency", "radius", "speed"])[["delta_abs", "delta_pct"]].mean().reset_index()

    dfs = [df_plot]

    columns = list(df_plot.drop(columns=["delta_abs", "delta_pct", x]).columns)
    for components in itertools.product(*[df_plot[c].unique() for c in columns]):
        record = {columns[i]: components[i] for i in range(len(columns))}
        record["delta_abs"] = 0
        record["delta_pct"] = 0
        record[x] = 0
        dfs.append(pd.DataFrame.from_records([record]))

    df_plot = pd.concat(dfs)
    fig = px.line(df_plot, x=x, y=y, color=color, facet_col=facet_col, markers=True,
                  title=title,
                  labels=dict(speed="Service speed (m/s)",
                              delta_abs="Difference in routing cost (%)",
                              delta="Difference in routing cost",
                              frequency="Frequency (s)",
                              radius="Feeder radius (m)"))
    if y == "delta_pct":
        fig.layout.yaxis.tickformat = ',.0%'
    return fig


impact_on_improved_routing_line_plot("radius", "frequency", "speed", "delta_pct", "Impact of feeder radius on improvement in routing cost").show()

impact_on_improved_routing_line_plot("radius", "frequency", "speed", "delta_abs", "Impact of feeder radius on improvement in routing cost").show()

In [ ]:
def impact_on_improved_routing_violin_plot(x, color, facet_col, y, title=None):
    fig = px.violin(df_routing_costs_comparison[df_routing_costs_comparison["delta_abs"] < 0],
        x=x, color=color, facet_col=facet_col, y=y,
        box=True,
        title=None
    )
    return fig

impact_on_improved_routing_violin_plot("radius", "frequency", "speed", "delta_abs").show()
impact_on_improved_routing_violin_plot("radius", "frequency", "speed", "delta_pct").show()